# 09 · Figures and descriptive tables
No training. Writes the manuscript figures to `results/paper/figures/` (PDF + 600-dpi PNG) and the descriptive tables
(dataset profile, selected hyperparameters, window length, class imbalance, cost) to `results/paper/tables/`.
Statistics and intervals come from notebook 10.

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 220, 'display.max_columns', 60, 'display.max_rows', 300)
from sdnids_bench.config import load_config, mode_params, active_datasets
cfg = load_config('..')
mp = mode_params(cfg)
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench.experiments import load_results, choose_best_L, ds_dir
from sdnids_bench import analysis as A, stats as S, figures as F, tables as TB, paper_figures as PF
DATASETS = active_datasets(cfg, 'B1')
PRIMARY = cfg['primary_dataset']
L = choose_best_L(cfg)                      # 8 in the paper (selected on validation data in B3)
PAPER = os.path.join(cfg['out_dir'], 'paper')
res = load_results(cfg, list(cfg['datasets']))
print('best L =', L); print(res.groupby(['dataset', 'exp']).size())
FIG = os.path.join(PAPER, 'figures'); TAB = os.path.join(PAPER, 'tables')
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)

## Figures 1–4: protocol diagram, class distribution, protocol inflation, per-class recall

In [ ]:
PF.fig_protocols(FIG)                                         # Figure "protocols"
profiles = {ds: pd.read_csv(os.path.join(ds_dir(cfg, ds), 'profile.csv')) for ds in DATASETS}
F.fig_class_distribution(profiles, FIG)                       # Figure 1
for ds in DATASETS:
    for LL in sorted({1, L}):
        F.fig_protocol_inflation(res[res.exp == 'B1'], FIG, ds, LL)
for ds in DATASETS:
    g = res[(res.exp == 'B1') & (res.dataset == ds) & (res.protocol == 'SC') & (res.task == 'multi') & (res.L == L)]
    classes = g['classes'].iloc[0]
    mat = g.groupby('model')[[f'recall_c{i}' for i in range(len(classes))]].mean()
    mat.columns = classes
    mat = mat.loc[g.groupby('model')['macro_f1'].mean().sort_values(ascending=False).index]
    n_test = {c: int(g[f'n_test_c{i}'].iloc[0]) for i, c in enumerate(classes)}
    F.fig_per_class_heatmap(mat, n_test, FIG, title=f'{ds}: per-class test recall (S-C, L = {L})', name=f'fig4_per_class_recall_{ds}')
# two-panel versions used in the manuscript
PF.stack_panels([os.path.join(FIG, f'fig2_protocol_inflation_{ds}_L{L}.png') for ds in DATASETS],
                [f'({chr(97 + i)}) {ds}' for i, ds in enumerate(DATASETS)], os.path.join(FIG, 'fig2_inflation.png'))
PF.stack_panels([os.path.join(FIG, f'fig4_per_class_recall_{ds}.png') for ds in DATASETS],
                [f'({chr(97 + i)}) {ds}' for i, ds in enumerate(DATASETS)], os.path.join(FIG, 'fig4_per_class.png'))

## Figures 5–9 and S1: confusion matrices, window length, imbalance, latency, rank agreement, learning curves

In [ ]:
from sdnids_bench.training import confusion
g_all = res[(res.exp == 'B1') & (res.dataset == PRIMARY) & (res.protocol == 'SC') & (res.task == 'multi') & (res.L == L)]
classes = g_all['classes'].iloc[0]
bn, bt = A.best_neural_and_tabular(res, PRIMARY, 'multi', L)   # chosen on validation macro-F1
mats = {}
for m in [x for x in (bt, bn) if x]:
    cm = None
    for k in g_all[g_all.model == m]['key']:
        z = np.load(os.path.join(ds_dir(cfg, PRIMARY), 'preds', k + '.npz'))
        c1 = confusion(z['y'].astype(int), z['P'].astype(float), len(classes))
        cm = c1 if cm is None else cm + c1
    mats[f"{m} ({(g_all.model == m).sum()} seeds pooled)"] = cm
F.fig_confusions(mats, classes, FIG, name=f'fig8_confusion_{PRIMARY}')
for ds in active_datasets(cfg, 'B3'):
    F.fig_context_length(res[res.exp == 'B3'], FIG, ds)
for ds in active_datasets(cfg, 'B4'):
    F.fig_imbalance_tradeoff(res[res.exp == 'B4'], FIG, ds)
for ds in active_datasets(cfg, 'B5'):
    F.fig_pareto(res[res.exp == 'B5'], FIG, ds)
M, _ = A.rank_agreement(res, DATASETS, {ds: L for ds in DATASETS})
if M.shape[1] >= 2:
    F.fig_rank_agreement(M, FIG)
hist = {}
g0 = g_all[g_all.seed == 0]
for m in ['MLP', 'CNN1D', 'LSTM', 'GRU', 'BiLSTM', 'CNN-LSTM', 'Transformer']:
    r = g0[g0.model == m]
    if len(r):
        p = os.path.join(ds_dir(cfg, PRIMARY), 'histories', r['key'].iloc[0] + '.json')
        if os.path.exists(p):
            hist[m] = json.load(open(p))
if hist:
    F.fig_learning_curves(hist, FIG)
print(sorted(os.listdir(FIG)))

## Descriptive tables

In [ ]:
T = {}
T['table_dataset_profile'] = (TB.table_dataset_profile(profiles), 'Flow records per class before and after deduplication.')
T['table_hyperparameters'] = (TB.table_hyperparams(cfg, DATASETS), f"Selected hyperparameters ({mp['trials']} Optuna trials per model, task and dataset; S-C validation macro-F1, L = {cfg['tune_L']}).")
for ds in DATASETS:
    for t in cfg['tasks']:
        T[f'table_main_{ds}_{t}'] = (TB.table_main_results(res, None, ds, t, L), f'{ds}, {t}: test metrics under S-C (L = {L}), mean ± SD over seeds.')
for ds in active_datasets(cfg, 'B3'):
    T[f'table_context_{ds}'] = (TB.table_context(res, ds), f'{ds}: test macro-F1 by window length (mean ± SD).')
for ds in active_datasets(cfg, 'B4'):
    cls = res[(res.dataset == ds) & (res.task == 'multi')]['classes'].iloc[0]
    T[f'table_imbalance_{ds}'] = (TB.table_imbalance(res, ds, cls), f'{ds}: class-imbalance strategies (S-C, L = {L}).')
for ds in active_datasets(cfg, 'B5'):
    T[f'table_cost_{ds}'] = (TB.table_cost(res, ds), f'{ds}: computational cost on CPU; latency is the median per run, averaged over seeds.')
TB.export_tables(T, TAB)
print(sorted(os.listdir(TAB)))